In [1]:
# ============================================================
# NB_INGST_BronzeSFTP_SalesOrderHeader
# Capa: Bronze | Origen: SFTP | Tabla: SalesOrderHeader
# Carga INCREMENTAL — solo archivos nuevos por fecha
# Todo desde config — sin hardcodeos (cumple TP Final)
# ============================================================
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from datetime import datetime
import re

# ============================================================
# IDs de workspace y lakehouses — sin hardcodear (TP)
# El pipeline setea lh_landing como defaultLakehouse de este notebook.
# ============================================================
# lakehouse.get(nombre) resuelve por nombre desde el workspace.
# Funciona en ejecución manual y desde pipeline. WS_ID nunca hardcodeado.
_lh_landing = notebookutils.lakehouse.get('lh_landing')
WS_ID       = _lh_landing.workspaceId
ART_LANDING = _lh_landing.id
# ART_BRONZE no necesario: Bronze escribe con saveAsTable (nombre lógico)

# Leer config desde lh_config — nunca hardcodear parámetros
config = spark.sql("""
    SELECT *
    FROM lh_config.dbo.origin_bronze_silver
    WHERE origen       = 'SFTP'
      AND nombre_tabla = 'SalesOrderHeader'
      AND activo       = 1
    LIMIT 1
""").collect()[0]

nombre_tabla  = config['nombre_tabla']              # SalesOrderHeader
origen        = config['origen']                    # SFTP
path_relativo = config['ubicacion_relativa']        # Files/TP_AdventureWorks/Sales
ultimo_valor  = config['ultimo_valor_incremental'] or '00000000'
params_inc    = config['parametros_incrementales']  # fecha_archivo
tipo_carga    = config['tipo_carga']                # incremental
pks           = config['pks'].split(',') if config['pks'] else []
separador_csv = config['separador_csv'] if config['separador_csv'] else ','

# Path ABFS construido dinámicamente — sin hardcodear rutas (TP)
# Fabric requiere que el path empiece con 'Files/' para acceder a archivos.
# Se normaliza desde config — sin hardcodear la ruta.
_path = path_relativo.strip('/')
if not _path.startswith('Files/'):
    _path = 'Files/' + _path
abfs_origen = (
    f'abfss://{WS_ID}@onelake.dfs.fabric.microsoft.com'
    f'/{ART_LANDING}/{_path}'
)

tabla_destino = f'lh_bronze.{origen}.{nombre_tabla}'

print('✅ Config cargada')
print(f'   nombre_tabla  : {nombre_tabla}')
print(f'   origen        : {origen}')
print(f'   tipo_carga    : {tipo_carga}')
print(f'   ultimo_valor  : {ultimo_valor}')
print(f'   params_inc    : {params_inc}')
print(f'   separador_csv : {separador_csv}')
print(f'   path_relativo : {path_relativo}')
print(f'   abfs_origen   : {abfs_origen}')


StatementMeta(, c7a84d19-7990-4465-8ba7-b27bbd2d847a, 3, Finished, Available, Finished, False)

✅ Config cargada
   nombre_tabla  : SalesOrderHeader
   origen        : SFTP
   tipo_carga    : incremental
   ultimo_valor  : 20260221
   params_inc    : fecha_archivo
   separador_csv : ;
   path_relativo : TP_AdventureWorks/Sales
   abfs_origen   : abfss://None@onelake.dfs.fabric.microsoft.com/f27e2853-1dca-44d9-94f8-08477a5836ff/TP_AdventureWorks/Sales


In [4]:
# ============================================================
# BONUS: Lógica de Reproceso
# ============================================================
try:
    fecha_inicio_reproceso = notebookutils.notebook.getParameter('fecha_inicio_reproceso', '')
    fecha_fin_reproceso    = notebookutils.notebook.getParameter('fecha_fin_reproceso', '')
except Exception:
    fecha_inicio_reproceso = ''
    fecha_fin_reproceso    = ''

modo_reproceso = bool(fecha_inicio_reproceso and fecha_fin_reproceso)
if modo_reproceso:
    print(f"REPROCESO Bronze SOH: {fecha_inicio_reproceso} → {fecha_fin_reproceso}")
    spark.sql(
        "DELETE FROM lh_bronze." + origen + "." + nombre_tabla +
        " WHERE " + params_inc + " >= '" + fecha_inicio_reproceso + "'" +
        " AND "   + params_inc + " <= '" + fecha_fin_reproceso    + "'"
    )
    spark.sql(
        "UPDATE lh_config.dbo.origin_bronze_silver" +
        " SET ultimo_valor_incremental = '" + fecha_inicio_reproceso + "'" +
        " WHERE nombre_tabla = '" + nombre_tabla + "' AND origen = '" + origen + "'"
    )
    ultimo_valor = fecha_inicio_reproceso
    print("   Bronze y watermark reseteados para reproceso")
else:
    print("Modo incremental normal")


StatementMeta(, c7a84d19-7990-4465-8ba7-b27bbd2d847a, 6, Finished, Available, Finished, False)

Modo incremental normal


In [5]:
# ============================================================
# Listar archivos NUEVOS en origen SFTP
# Incremental: fecha del nombre > ultimo_valor_incremental
# Formato esperado: NombreTabla_AAAAMMDD.csv
# ============================================================

archivos_disponibles = notebookutils.fs.ls(abfs_origen)

archivos_nuevos = []
for f in archivos_disponibles:
    match = re.search(r'_(\d{8})\.csv$', f.name)
    if match:
        fecha_archivo = match.group(1)
        if fecha_archivo > ultimo_valor:
            archivos_nuevos.append({
                'path'  : f.path,
                'name'  : f.name,
                'fecha' : fecha_archivo
            })

print(f'📄 Disponibles : {len(archivos_disponibles)}')
print(f'📄 Nuevos      : {len(archivos_nuevos)} (posterior a {ultimo_valor})')
for a in archivos_nuevos:
    print(f'   → {a["name"]} (fecha: {a["fecha"]})')

if not archivos_nuevos:
    print('ℹ️  Sin archivos nuevos — pipeline finaliza aquí')
    notebookutils.notebook.exit('sin_archivos_nuevos')


StatementMeta(, c7a84d19-7990-4465-8ba7-b27bbd2d847a, 7, Finished, Available, Finished, False)

Py4JJavaError: An error occurred while calling z:notebookutils.fs.ls.
: Operation failed: "Bad Request", 400, GET, http://onelake.dfs.fabric.microsoft.com/None?upn=false&resource=filesystem&maxResults=5000&directory=f27e2853-1dca-44d9-94f8-08477a5836ff/TP_AdventureWorks/Sales&timeout=90&recursive=false, FriendlyNameSupportDisabled, "Request Failed with WorkspaceId and ArtifactId should be either valid Guids or valid Names"
	at org.apache.hadoop.fs.azurebfs.services.AbfsRestOperation.completeExecute(AbfsRestOperation.java:231)
	at org.apache.hadoop.fs.azurebfs.services.AbfsRestOperation.lambda$execute$0(AbfsRestOperation.java:191)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.trackDurationOfInvocation(IOStatisticsBinding.java:464)
	at org.apache.hadoop.fs.azurebfs.services.AbfsRestOperation.execute(AbfsRestOperation.java:189)
	at org.apache.hadoop.fs.azurebfs.services.AbfsClient.listPath(AbfsClient.java:311)
	at org.apache.hadoop.fs.azurebfs.AzureBlobFileSystemStore.listS

In [6]:
# ============================================================
# Leer CSVs nuevos, castear TODO a string, guardar Bronze
# Requisito TP Bronze: todo string + columnas de auditoría
# LIMIT 1000 obligatorio durante desarrollo (TP)
# FIX: separador desde config (no hardcodeado)
# FIX: limpiar nombres de columna antes de guardar
# ============================================================

fecha_carga = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
total_filas = 0

for archivo in archivos_nuevos:
    print(f'\n⚙️  Procesando: {archivo["name"]}')

    df_raw = (
        spark.read
        .option('header', 'true')
        .option('inferSchema', 'false')
        # FIX: separador desde config — no hardcodeado
        .option('sep', separador_csv)
        .csv(archivo['path'])
        .limit(1000)  # ← LIMIT 1000 obligatorio TP
    )

    print(f'   Columnas raw : {len(df_raw.columns)}')
    print(f'   Filas raw    : {df_raw.count()}')

    # Verificar que el separador funcionó
    if len(df_raw.columns) < 5:
        raise Exception(
            f'❌ Solo {len(df_raw.columns)} columna(s) detectada(s).\n'
            f'   El separador\'{separador_csv}\' no es correcto.\n'
            f'   Actualizar separador_csv en config y reintentar.'
        )

    # Castear TODO a string (requisito Bronze TP)
    # Limpiar nombres de columna (quitar caracteres especiales)
    df_bronze = df_raw.select([
        F.col(c).cast(StringType())
         .alias(re.sub(r'[ ,;{}()\n\t=]', '_', c.strip()))
        for c in df_raw.columns
    ])

    # Columnas de auditoría (obligatorias en Bronze según TP)
    df_bronze = (
        df_bronze
        .withColumn('fecha_carga',    F.lit(fecha_carga))
        .withColumn('nombre_archivo', F.lit(archivo['name']))
        .withColumn('fecha_archivo',  F.lit(archivo['fecha']))
    )

    (
        df_bronze.write
        .format('delta')
        .mode('append')
        .option('mergeSchema', 'true')
        .saveAsTable(tabla_destino)
    )

    filas = df_bronze.count()
    total_filas += filas
    print(f'   ✅ {filas} filas → {tabla_destino}')

print(f'\n✅ TOTAL filas cargadas : {total_filas}')
print(f'✅ Tabla destino        : {tabla_destino}')


StatementMeta(, c7a84d19-7990-4465-8ba7-b27bbd2d847a, 8, Finished, Available, Finished, False)

NameError: name 'archivos_nuevos' is not defined

In [7]:
# ============================================================
# Actualizar watermark en config
# MAX(fecha) de los archivos procesados en esta ejecución
# Variables desde config — sin hardcodeos
# ============================================================

if archivos_nuevos:
    max_fecha = max([a['fecha'] for a in archivos_nuevos])
    spark.sql(f"""
        UPDATE lh_config.dbo.origin_bronze_silver
        SET ultimo_valor_incremental = '{max_fecha}'
        WHERE origen       = '{origen}'
          AND nombre_tabla = '{nombre_tabla}'
    """)
    print(f'✅ Watermark actualizado: {ultimo_valor} → {max_fecha}')
else:
    print('ℹ️  Sin archivos nuevos — watermark sin cambios')


StatementMeta(, c7a84d19-7990-4465-8ba7-b27bbd2d847a, 9, Finished, Available, Finished, False)

NameError: name 'archivos_nuevos' is not defined

In [8]:
# ============================================================
# Verificar resultado en Bronze
# ============================================================

display(spark.sql(f"""
    SELECT
        nombre_archivo,
        fecha_archivo,
        fecha_carga,
        COUNT(*) AS cant_filas
    FROM {tabla_destino}
    GROUP BY nombre_archivo, fecha_archivo, fecha_carga
    ORDER BY fecha_archivo
"""))

total = spark.sql(
    f'SELECT COUNT(*) as cnt FROM {tabla_destino}'
).collect()[0]['cnt']
print(f'✅ Total filas en Bronze ({tabla_destino}): {total}')


StatementMeta(, c7a84d19-7990-4465-8ba7-b27bbd2d847a, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 02a98874-ee67-4cc1-86d2-0c529b6513c5)

✅ Total filas en Bronze (lh_bronze.SFTP.SalesOrderHeader): 4000
